In [1]:
from collections import defaultdict
import glob
import json
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import pandas as pd
from scipy.stats import wilcoxon
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
paths = glob.glob("../../intermediates/reannotation/*/single_isoform/*lengths.tsv")

def iter_frames(paths):
    for path in paths:
        dataset = path.split("/")[-3]
        genome = path.split("/")[-1].replace("_lengths.tsv", "")

        df = pd.read_csv(
            path,
            sep="\t",
            header=0,
            usecols=["protein_id", "length"],
            dtype={"protein_id": "string", "length": "int32"},
            engine="c",
            low_memory=False,
        )
        df["dataset"] = dataset
        df["genome"] = genome
        yield df

lengths_df = pd.concat(iter_frames(paths), ignore_index=True, copy=False)

In [ ]:
plot_data = lengths_df.dropna(subset=["length", "dataset"]).copy()
plot_data["length"] = pd.to_numeric(plot_data["length"], errors="coerce")
plot_data = plot_data.dropna(subset=["length"])

hue_order = ["reference", "geneml"]
palette = ["#B4B4B4", "#FF9B59"]
dataset_labels = {"reference": "original", "geneml": "geneML"}

fig, ax = plt.subplots(figsize=(8, 5.4))
sns.ecdfplot(
    data=plot_data,
    x="length",
    hue="dataset",
    hue_order=hue_order,
    palette=palette,
    linewidth=2,
    ax=ax,
)

ax.set_xscale("log")
ax.set_xlim(left=10)
ax.set_xlabel("Protein length (aa)")
ax.set_ylabel("Cumulative fraction of genes")
ax.set_title("Protein length distributions")

summary = (
    plot_data.groupby("dataset")["length"]
    .agg(
        median="median",
        long_frac=lambda s: (s > 1000).mean(),
        n="size",
    )
    .reindex(hue_order)
)

for dataset, color in zip(hue_order, palette):
    median = summary.loc[dataset, "median"]
    ax.axvline(median, color=color, linestyle="--", linewidth=1.5, alpha=0.85)

if ax.get_legend() is not None:
    ax.get_legend().set_title("Annotations")
    for text in ax.get_legend().get_texts():
        current = text.get_text()
        text.set_text(dataset_labels.get(current, current))

annotation_lines = []
for dataset in hue_order:
    row = summary.loc[dataset]
    label = dataset_labels.get(dataset, dataset)
    annotation_lines.append(
        f"{label}: N={int(row['n']):,}, median={row['median']:.0f} aa, P(length>1000)={row['long_frac']:.1%}"
    )

fig.subplots_adjust(bottom=0.20)
fig.text(
    0.1,
    0.04,
    "\n".join(annotation_lines),
    ha="left",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.8", alpha=0.9),
)

fig.tight_layout(rect=(0, 0.08, 1, 1))
fig.savefig("../../figures/Figure_S5.svg")

In [2]:
busco_stats = defaultdict(list)
for path in glob.glob("../../intermediates/reannotation/*/busco/*.json"):
    with open(path, 'r') as s:
        data = json.load(s)
        path_in = data["parameters"]["in"]
        dataset = path_in.split('/')[-3]
        accession = str.join('_', path_in.split('/')[-1].split('_')[:2])
        busco_stats["accession"].append(accession)
        busco_stats["dataset"].append(dataset)
        busco_stats["completeness"].append(data["results"]["Complete percentage"])
        busco_stats["fragmented"].append(data["results"]["Fragmented percentage"])
        busco_stats["missing"].append(data["results"]["Missing percentage"])
        busco_stats["multi_copy"].append(data["results"]["Multi copy percentage"])
busco_df = pd.DataFrame(busco_stats)

In [4]:
busco_df.groupby("dataset")["completeness"].describe()

,count,mean,std,min,25%,50%,75%,max
dataset,,,,,,,,
geneml,761.0,92.950854,12.528574,4.5,92.8,97.2,98.8,99.6
reference,761.0,91.635742,13.653523,1.3,91.1,96.3,98.3,99.7


In [ ]:
def get_annotation_counts(tsv_file):
    counts = defaultdict(int)
    annotations = set()
    last_gene_id = None
    with open(tsv_file, 'r') as s:
        next(s)
        for line in s:
            fields = line.strip().split("\t")
            gene_id = fields[0]
            annotation = fields[3]
            if gene_id == last_gene_id:
                annotations.add(annotation)
            else:
                if last_gene_id is not None:
                    if "Complete" in annotations:
                        counts["Complete"] += 1
                    elif "Partial" in annotations:
                        counts["Partial"] += 1
                    else:
                        counts["None"] += 1
                annotations = {annotation}
            last_gene_id = gene_id

    if last_gene_id is not None:
        if "Complete" in annotations:
            counts["Complete"] += 1
        elif "Partial" in annotations:
            counts["Partial"] += 1
        else:
            counts["None"] += 1

    return counts

In [ ]:
paths = glob.glob("../../intermediates/reannotation/*/pfam/*annotations.tsv")
records = []

for path in paths:
    counts = get_annotation_counts(path)
    genome = path.split("/")[-1].replace("_annotations.tsv", "")
    dataset = path.split("/")[-3]

    for annotation_type in ["partial", "complete", "none"]:
        records.append({
            "type": annotation_type,
            "count": counts.get(annotation_type.capitalize(), 0),
            "genome": genome,
            "dataset": dataset,
        })

annotation_df = pd.DataFrame(records)

In [ ]:
genome_details = pd.read_csv("../../data/training_set_details.tsv", sep="\t")

annotation_df["accession"] = annotation_df["genome"].str.split("_").str[0:2].str.join("_")
annotation_df = annotation_df.merge(genome_details, on="accession", how="left")
annotation_df["phylum_category"] = annotation_df["phylum"].where(
    annotation_df["phylum"].isin(["Ascomycota", "Basidiomycota"]),
    "Other",
)
phylum_order = ["Ascomycota", "Basidiomycota", "Other"]
annotation_df["phylum_category"] = pd.Categorical(
    annotation_df["phylum_category"],
    categories=phylum_order,
    ordered=True,
)

In [ ]:
dataset_table = (
    annotation_df.groupby("dataset")["count"]
    .agg(
        median="median",
        mean="mean",
        min="min",
        max="max",
        q1=lambda s: s.quantile(0.25),
        q3=lambda s: s.quantile(0.75),
    )
)

dataset_table.to_csv("../../tables/Table_S9.tsv", sep="\t", index=True)

In [ ]:
table_df = (
    annotation_df.groupby(["dataset", "type", "phylum_category"], as_index=False)["count"]
    .sum()
)

totals_df = (
    table_df.groupby(["dataset", "type"], as_index=False)["count"]
    .sum()
    .assign(phylum_category="All")
)

table_long = pd.concat([table_df, totals_df], ignore_index=True)

phylum_order = ["Ascomycota", "Basidiomycota", "Other", "All"]
dataset_order = ["original", "geneml"]
type_order = ["none", "partial", "complete"]

dataset_phylum_table = (
    table_long.pivot_table(
        index=["phylum_category", "dataset"],
        columns="type",
        values="count",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=type_order, fill_value=0)
    .reset_index()
)

dataset_phylum_table.columns.name = None
dataset_phylum_table["dataset"] = dataset_phylum_table["dataset"].replace({"reference": "original"})
dataset_phylum_table["phylum_category"] = pd.Categorical(
    dataset_phylum_table["phylum_category"],
    categories=phylum_order,
    ordered=True,
)
dataset_phylum_table["dataset"] = pd.Categorical(
    dataset_phylum_table["dataset"],
    categories=dataset_order,
    ordered=True,
)
dataset_phylum_table = dataset_phylum_table.sort_values(["phylum_category", "dataset"]).reset_index(drop=True)

dataset_phylum_table[type_order] = dataset_phylum_table[type_order].astype(int)
dataset_phylum_table = dataset_phylum_table.rename(
    columns={
        "phylum_category": "Phylum",
        "dataset": "Annotation dataset",
        "none": "Genes without PFAM domains",
        "partial": "Genes with partial PFAM domains",
        "complete": "Genes with complete PFAM domains",
    }
)

dataset_phylum_table.to_csv("../../tables/Table_S10.tsv", sep="\t", index=False)

In [ ]:
plot_df = (
    annotation_df.groupby(["type", "dataset", "phylum_category"], as_index=False)["count"]
    .sum()
)

phylum_titles = {
    "Ascomycota": "Ascomycota",
    "Basidiomycota": "Basidiomycota",
    "Other": "Other phyla",
}
type_labels = {
    "none": "None",
    "partial": "Partial",
    "complete": "Complete",
}
dataset_labels = {
    "reference": "original",
    "geneml": "geneML",
}

type_order = ["none", "partial", "complete"]
plot_df["type_label"] = plot_df["type"].map(type_labels)
label_order = [type_labels[t] for t in type_order]

g = sns.catplot(
    data=plot_df,
    x="count",
    y="type_label",
    hue="dataset",
    hue_order=["reference", "geneml"],
    col="phylum_category",
    kind="bar",
    order=label_order,
    height=5,
    aspect=0.85,
    sharex=False,
    palette=["#B4B4B4", "#FF9B59"],
    saturation=1,
)

g.set_axis_labels("Number of genes", "PFAM domain annotation")

# Apply scientific notation and set subplot titles
for ax in g.axes.flat:
    ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))
    ax.tick_params(axis="both", labelsize=11)
    ax.xaxis.label.set_size(12)
    ax.yaxis.label.set_size(12)
    phylum = ax.get_title().split(" = ", 1)[-1]
    ax.set_title(phylum_titles.get(phylum, phylum), fontsize=12)

# Rename legend labels, then place legend below plots (left-aligned)
if g._legend is not None:
    for text in g._legend.texts:
        current = text.get_text()
        text.set_text(dataset_labels.get(current, current))

sns.move_legend(
    g,
    "lower left",
    bbox_to_anchor=(0.06, -0.06),
    ncol=2,
    title="Annotation",
    frameon=False,
)

if g._legend is not None:
    g._legend.get_title().set_fontsize(13)
    g._legend.get_title().set_ha("left")
    g._legend._legend_box.align = "left"
    for text in g._legend.texts:
        text.set_fontsize(12)

g.fig.subplots_adjust(bottom=0.22)
g.fig.savefig("../../figures/Figure_5A.svg", bbox_inches="tight")

In [ ]:
def get_bin_counts(tsv_file):
    gene_counts = defaultdict(int)
    pfam_bin_counts = defaultdict(int)

    with open(tsv_file, "r") as s:
        next(s)
        for line in s:
            fields = line.rstrip("\n").split("\t")
            if len(fields) < 4:
                continue

            pfam_ids = fields[1].strip()
            bin_id = fields[3].strip()
            gene_counts[bin_id] += 1

            # Count PFAM IDs once per row, keeping each ID linked to its bin.
            if pfam_ids and pfam_ids != "NA":
                unique_pfams = {pfam.strip() for pfam in pfam_ids.split(",") if pfam.strip()}
                for pfam_id in unique_pfams:
                    pfam_bin_counts[(bin_id, pfam_id)] += 1

    return gene_counts, pfam_bin_counts

In [ ]:
paths = glob.glob("../../intermediates/reannotation/*/pfam/*functions.tsv")
bin_records = []
pfam_records = []

for path in paths:
    gene_counts, pfam_bin_counts = get_bin_counts(path)
    genome = path.split("/")[-1].replace("_functions.tsv", "")
    dataset = path.split("/")[-3]

    for bin_id in list(range(1, 9)) + [100]:
        bin_key = str(bin_id)
        bin_records.append({
            "type": bin_id,
            "count": gene_counts.get(bin_key, 0),
            "genome": genome,
            "dataset": dataset,
        })

    for (bin_key, pfam_id), pfam_count in pfam_bin_counts.items():
        pfam_records.append({
            "type": int(bin_key),
            "pfam_id": pfam_id,
            "pfam_count": pfam_count,
            "genome": genome,
            "dataset": dataset,
        })

functions_df = pd.DataFrame(bin_records)
pfam_counts_df = pd.DataFrame(pfam_records)

In [ ]:
go_bins = pd.read_csv("../../data/go_bins.tsv", sep="\t")
functions_df = functions_df.merge(go_bins, left_on="type", right_on="bin_id", how="left")

In [ ]:
functional_bin_table = (
    functions_df.groupby(["bin_name", "dataset"], as_index=False)["count"]
    .sum()
    .replace({"reference": "original"})
)

functional_bin_table = (
    functional_bin_table.pivot_table(
        index="bin_name",
        columns="dataset",
        values="count",
        aggfunc="sum",
        fill_value=0,
    )
    .reindex(columns=["original", "geneml"], fill_value=0)
    .reset_index()
    .rename(columns={"bin_name": "Functional bin"})
)

functional_bin_table.columns.name = None
functional_bin_table[["original", "geneml"]] = functional_bin_table[["original", "geneml"]].astype(int)
functional_bin_table.to_csv("../../tables/Table_S11.tsv", sep="\t", index=False)

In [ ]:
plot_df = (
    functions_df.groupby(["bin_name", "dataset"], as_index=False)["count"]
    .sum()
)

dataset_labels = {
    "reference": "original",
    "geneml": "geneML",
}

bin_order = (
    plot_df.groupby("bin_name")["count"]
    .sum()
    .sort_values(ascending=False)
    .index
    .tolist()
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(
    data=plot_df,
    x="count",
    y="bin_name",
    hue="dataset",
    hue_order=["reference", "geneml"],
    order=bin_order,
    palette=["#B4B4B4", "#FF9B59"],
    saturation=1,
    ax=ax,
)

ax.set_xlabel("Number of genes")
ax.set_ylabel("Functional bin")
ax.tick_params(axis="both", labelsize=11)
ax.xaxis.label.set_size(12)
ax.yaxis.label.set_size(12)
ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax.ticklabel_format(style="sci", axis="x", scilimits=(0, 0))

if ax.get_legend() is not None:
    ax.get_legend().remove()
fig.savefig("../../figures/Figure_5B.svg", bbox_inches="tight")

In [ ]:
# Per-genome paired PFAM counts for each bin/PFAM; exclude bin 100.
paired_counts = (
    pfam_counts_df[pfam_counts_df["type"] != 100]
    .pivot_table(
        index=["type", "pfam_id", "genome"],
        columns="dataset",
        values="pfam_count",
        aggfunc="sum",
        fill_value=0,
    )
    .reset_index()
)

if "reference" not in paired_counts.columns:
    paired_counts["reference"] = 0
if "geneml" not in paired_counts.columns:
    paired_counts["geneml"] = 0


def benjamini_hochberg(p_values):
    n = len(p_values)
    if n == 0:
        return []
    ranked = sorted(enumerate(p_values), key=lambda x: x[1])
    q = [1.0] * n
    prev = 1.0
    for rank in range(n, 0, -1):
        idx, p = ranked[rank - 1]
        value = min(prev, p * n / rank)
        q[idx] = value
        prev = value
    return q


rows = []
for (bin_type, pfam_id), grp in paired_counts.groupby(["type", "pfam_id"], sort=False):
    x = grp["reference"].astype(float)
    y = grp["geneml"].astype(float)
    diff = x - y

    if (diff == 0).all():
        p_value = 1.0
    else:
        p_value = wilcoxon(x, y, alternative="two-sided", zero_method="wilcox").pvalue

    rows.append({
        "Bin type": int(bin_type),
        "PFAM ID": pfam_id,
        "n paired genomes": int(len(grp)),
        "median original": float(x.median()),
        "median geneml": float(y.median()),
        "total diff (geneml-original)": float((y - x).sum()),
        "p_value": float(p_value),
    })

pfam_stats_table = pd.DataFrame(rows)
pfam_stats_table["q_value"] = benjamini_hochberg(pfam_stats_table["p_value"].tolist())
pfam_stats_table = pfam_stats_table[pfam_stats_table["q_value"] < 0.05]

In [ ]:
pfam_desc = {}
pfam_hmm_path = "../../databases/pfam/Pfam-A.hmm"

with open(pfam_hmm_path, "r", encoding="utf-8", errors="replace") as f:
    current_acc = None
    current_desc = None
    current_name = None

    for line in f:
        if line.startswith("NAME  "):
            current_name = line[6:].strip()
        elif line.startswith("ACC   "):
            current_acc = line[6:].strip().split(".", 1)[0]
        elif line.startswith("DESC  "):
            current_desc = line[6:].strip()
        elif line.strip() == "//":
            if current_acc:
                pfam_desc[current_acc] = current_desc or current_name or current_acc
            current_acc = None
            current_desc = None
            current_name = None

In [ ]:
pfam_stats_table = pfam_stats_table.sort_values("total diff (geneml-original)", ascending=False).reset_index(drop=True)
pfam_stats_summary = pfam_stats_table.head(20).copy()
pfam_stats_summary["PFAM description"] = (
    pfam_stats_summary["PFAM ID"]
    .str.replace(r"\.\d+$", "", regex=True)
    .map(pfam_desc)
    .fillna("NA")
)

pfam_stats_summary.to_csv("../../tables/Table_S12.tsv", sep="\t", index=False)